# ICAO Code to Timezone Mapping for WingX Metro Data
Enrich US-Domestic Metro flights with timezone and location data from ICAO mapping

## CELL 1: LOAD AND VALIDATE ICAO TIMEZONE MAPPING DATA

In [1]:
import pandas as pd
import numpy as np


print("🚀 Loading ICAO timezone mapping data...")

# Load ICAO timezone mapping from GCS
ICAO_MAPPING_PATH = "icao_timezone.csv"
airport_mapping = pd.read_csv(ICAO_MAPPING_PATH)

print(f"✓ Loaded {len(airport_mapping):,} airport records")
print(f"Columns: {airport_mapping.columns.tolist()}")
print(f"\nSample data:")
print(airport_mapping.head())

🚀 Loading ICAO timezone mapping data...
✓ Loaded 5,519 airport records
Columns: ['Unnamed: 0', 'ident', 'latitude_deg', 'longitude_deg', 'timezone']

Sample data:
   Unnamed: 0 ident  latitude_deg  longitude_deg         timezone
0       37473  K00C     37.203201    -107.869003   America/Denver
1       37474  K00F     45.470462    -105.457145   America/Denver
2       37475  K00M     31.953728     -89.235270  America/Chicago
3       37476  K00R     30.685900     -95.017899  America/Chicago
4       37477  K00V     38.945801    -104.570000   America/Denver


## CELL 2: LOAD US-DOMESTIC METRO FLIGHTS DATA

In [2]:
print("\n🚀 Loading US-Domestic Metro mission data...")

# Path to the Metro-mapped US Domestic data
FLIGHTS_PATH = "gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro.parquet"

# Load the dataset
df_wingx = pd.read_parquet(FLIGHTS_PATH)

# Terminology Standardization
rename_map = {
    'FromCluster': 'FromMetro',
    'ToCluster': 'ToMetro'
}
df_wingx = df_wingx.rename(columns=rename_map)

print(f"✓ Loaded {len(df_wingx):,} flight records")
print(f"Columns: {df_wingx.columns.tolist()}")
print(f"\nSample data:")
print(df_wingx[['FromAirport', 'ToAirport', 'FromMetro', 'ToMetro']].head())


🚀 Loading US-Domestic Metro mission data...
✓ Loaded 2,140,706 flight records
Columns: ['FlightDate_utc', 'Operator', 'FromAirport', 'FromState', 'ToAirport', 'ToState', 'Hours', 'aircraft_tailsign', 'aircraft_model', 'aircraft_segment', 'fuel_uplift_usg', 'FlightDate_ET', 'year', 'month', 'dow', 'hour', 'FromMetro', 'ToMetro']

Sample data:
  FromAirport ToAirport      FromMetro         ToMetro
0        KTEX      KBJC         Denver          Denver
1        KJAC      KSDL         Denver  Phoenix Valley
2        KORL      KCHO  North Florida             DMV
3        KFCM      KSDL    Minneapolis  Phoenix Valley
4        KSNA      KSLC       LA Basin          Denver


## CELL 3: ANALYZE MAPPING OVERLAP

In [3]:
print("\n" + "="*70)
print("ANALYZING ICAO TO TIMEZONE MAPPING")
print("="*70)

# Get all unique airports
all_airports = pd.concat([df_wingx['FromAirport']]).unique()

print(f"\nUnique ICAO codes in flights: {len(all_airports)}")
print(f"Unique ICAO codes in mapping: {len(airport_mapping['ident'].unique())}")

# Check overlap
icao_flights = set(all_airports)
icao_mapping = set(airport_mapping['ident'].unique())
overlap = icao_flights & icao_mapping

print(f"\nMapping overlap: {len(overlap)} codes ({len(overlap)/len(icao_flights)*100:.2f}%)")
print(f"Missing codes: {len(icao_flights - icao_mapping)} ({len(icao_flights - icao_mapping)/len(icao_flights)*100:.2f}%)")

if len(icao_flights - icao_mapping) > 0 and len(icao_flights - icao_mapping) <= 50:
    print(f"\nMissing airport codes:")
    print(sorted(icao_flights - icao_mapping))
elif len(icao_flights - icao_mapping) > 0:
    missing_codes = sorted(icao_flights - icao_mapping)
    print(f"\nMissing airport codes (showing first 30):")
    print(missing_codes[:30])


ANALYZING ICAO TO TIMEZONE MAPPING

Unique ICAO codes in flights: 2516
Unique ICAO codes in mapping: 5519

Mapping overlap: 2418 codes (96.10%)
Missing codes: 98 (3.90%)

Missing airport codes (showing first 30):
['K1K1', 'K1K8', 'K59B', 'K6B0', 'K6N5', 'K8D4', 'K91C', 'K92C', 'KADF', 'KAPS', 'KBAC', 'KBCR', 'KBLD', 'KBRG', 'KBVU', 'KBYL', 'KBZA', 'KC56', 'KC59', 'KCCC', 'KCFO', 'KCNI', 'KCPF', 'KCPP', 'KCQF', 'KCVC', 'KCZQ', 'KDNA', 'KDUB', 'KDWX']


In [4]:
print(missing_codes)

['K1K1', 'K1K8', 'K59B', 'K6B0', 'K6N5', 'K8D4', 'K91C', 'K92C', 'KADF', 'KAPS', 'KBAC', 'KBCR', 'KBLD', 'KBRG', 'KBVU', 'KBYL', 'KBZA', 'KC56', 'KC59', 'KCCC', 'KCFO', 'KCNI', 'KCPF', 'KCPP', 'KCQF', 'KCVC', 'KCZQ', 'KDNA', 'KDUB', 'KDWX', 'KEBA', 'KEHF', 'KETX', 'KFBK', 'KFHB', 'KFIN', 'KFPY', 'KFWB', 'KFYG', 'KGDA', 'KGDK', 'KGPC', 'KGZL', 'KHDL', 'KHMP', 'KHRF', 'KHTF', 'KHYP', 'KIIU', 'KIKW', 'KIUA', 'KIYA', 'KJAQ', 'KJCA', 'KJHN', 'KJPX', 'KJQD', 'KJRA', 'KJRO', 'KJSY', 'KJTC', 'KJVW', 'KK78', 'KLUV', 'KMAN', 'KMC1', 'KMMM', 'KMNE', 'KMQO', 'KMUF', 'KMXL', 'KN69', 'KNHZ', 'KNLD', 'KOZS', 'KPCD', 'KRCE', 'KRCV', 'KREG', 'KRGA', 'KRTS', 'KRVF', 'KSCA', 'KSCR', 'KSJS', 'KSVM', 'KSXU', 'KSYM', 'KT89', 'KTXW', 'KU64', 'KUCA', 'KVBW', 'KX61', 'KXNX', 'KY63', 'KZ15', 'KZQA']


In [5]:
df_wingx[df_wingx["FromAirport"]=="K00R"]

,FlightDate_utc,Operator,FromAirport,FromState,ToAirport,ToState,Hours,aircraft_tailsign,aircraft_model,aircraft_segment,fuel_uplift_usg,FlightDate_ET,year,month,dow,hour,FromMetro,ToMetro
1144524,2025-02-12T20:42:00.000Z,Free Jumper LLC,K00R,None,KSRQ,FL,1.75,N624MB,Citation CJ4 Gen2,Light Jet,336.0,2025-02-12 15:42:00-05:00,2025,2,Wednesday,15,OTHER_METRO,South Florida


In [6]:
airport_mapping[airport_mapping["ident"]=="K00R"]

,Unnamed: 0,ident,latitude_deg,longitude_deg,timezone
3,37476,K00R,30.6859,-95.017899,America/Chicago


## CELL 4: CREATE ICAO LOOKUP AND ENRICH FLIGHTS

In [7]:
print("\n" + "="*70)
print("ENRICHING FLIGHTS WITH TIMEZONE AND LOCATION DATA")
print("="*70)

# Create lookup dictionary
airport_lookup = airport_mapping.set_index('ident').to_dict('index')

# Enrich flights data
enriched_flights = df_wingx.copy()

# Map timezone and coordinates for FromAirport
enriched_flights['FromAirport_timezone'] = enriched_flights['FromAirport'].apply(
    lambda x: airport_lookup.get(x, {}).get('timezone') if x in airport_lookup else None
)
enriched_flights['FromAirport_latitude'] = enriched_flights['FromAirport'].apply(
    lambda x: airport_lookup.get(x, {}).get('latitude_deg') if x in airport_lookup else None
)
enriched_flights['FromAirport_longitude'] = enriched_flights['FromAirport'].apply(
    lambda x: airport_lookup.get(x, {}).get('longitude_deg') if x in airport_lookup else None
)


print(f"\nEnriched shape: {enriched_flights.shape}")
print(f"Original columns: {len(df_wingx.columns)}")
print(f"New columns added: {len(enriched_flights.columns) - len(df_wingx.columns)}")
print(f"New column names: {[col for col in enriched_flights.columns if col not in df_wingx.columns]}")

print("\nSample enriched data:")
sample_cols = ['FromAirport', 'ToAirport', 'FromAirport_timezone', 'FromMetro', 'ToMetro']
print(enriched_flights[sample_cols].head(10))


ENRICHING FLIGHTS WITH TIMEZONE AND LOCATION DATA

Enriched shape: (2140706, 21)
Original columns: 18
New columns added: 3
New column names: ['FromAirport_timezone', 'FromAirport_latitude', 'FromAirport_longitude']

Sample enriched data:
  FromAirport ToAirport FromAirport_timezone      FromMetro         ToMetro
0        KTEX      KBJC       America/Denver         Denver          Denver
1        KJAC      KSDL       America/Denver         Denver  Phoenix Valley
2        KORL      KCHO     America/New_York  North Florida             DMV
3        KFCM      KSDL      America/Chicago    Minneapolis  Phoenix Valley
4        KSNA      KSLC  America/Los_Angeles       LA Basin          Denver
5        KIAD      KPNE     America/New_York            DMV    Philadelphia
6        KJAC      KPIT       America/Denver         Denver      Pittsburgh
7        KEGE      KSAN       America/Denver         Denver        LA Basin
8        KSSI      KBED     America/New_York  North Florida          Boston
9

## CELL 5: COVERAGE ANALYSIS

In [8]:
print("\n" + "="*70)
print("COVERAGE ANALYSIS")
print("="*70)

from_with_tz = enriched_flights['FromAirport_timezone'].notna().sum()

print(f"\nTotal flight records: {len(enriched_flights):,}")
print(f"FromAirport records with timezone: {from_with_tz:,} ({from_with_tz/len(enriched_flights)*100:.2f}%)")

print(f"\nUnique airports with timezone data:")
print(f"  FromAirports: {enriched_flights[enriched_flights['FromAirport_timezone'].notna()]['FromAirport'].nunique()}")

# Timezone distribution
print(f"\nTimezone distribution:")
print(enriched_flights['FromAirport_timezone'].value_counts().head(10))


COVERAGE ANALYSIS

Total flight records: 2,140,706
FromAirport records with timezone: 2,125,312 (99.28%)

Unique airports with timezone data:
  FromAirports: 2418

Timezone distribution:
FromAirport_timezone
America/New_York                860147
America/Chicago                 666841
America/Los_Angeles             281246
America/Denver                  166238
America/Phoenix                  55263
America/Detroit                  40087
America/Indiana/Indianapolis     29055
America/Boise                    17308
America/Kentucky/Louisville       8420
America/Menominee                  422
Name: count, dtype: int64


## CELL 6: SAVE ENRICHED DATA TO GCS

In [9]:
print("\n" + "="*70)
print("SAVING ENRICHED DATA")
print("="*70)

# Save enriched flights to GCS
output_path = "wingx_timezone.csv"
enriched_flights.to_parquet(output_path)

print(f"\n[SAVED] Enriched flights saved to GCS")
print(f"Path: {output_path}")
print(f"Records: {len(enriched_flights):,}")
print(f"Columns: {len(enriched_flights.columns)}")


SAVING ENRICHED DATA

[SAVED] Enriched flights saved to GCS
Path: wingx_timezone.csv
Records: 2,140,706
Columns: 21


## CELL 7: FINAL SUMMARY

In [10]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\nData sources:")
print(f"  - Flights: {len(enriched_flights):,} records")
print(f"  - Airports: {len(airport_mapping):,} records")
print(f"  - Unique ICAO codes in flights: {len(all_airports)}")

print(f"\nMapping results:")
print(f"  - ICAO codes matched: {len(overlap)} ({len(overlap)/len(icao_flights)*100:.2f}%)")
print(f"  - Coverage: {from_with_tz/len(enriched_flights)*100:.2f}% of flights enriched")

print(f"\nEnriched dataset:")
print(f"  - Flights with both timezones: {from_with_tz:,} ({from_with_tz/len(enriched_flights)*100:.2f}%)")
print(f"  - New columns: 6 (timezone, latitude, longitude for both airports)")
print(f"  - Output path: {output_path}")


FINAL SUMMARY

Data sources:
  - Flights: 2,140,706 records
  - Airports: 5,519 records
  - Unique ICAO codes in flights: 2516

Mapping results:
  - ICAO codes matched: 2418 (96.10%)
  - Coverage: 99.28% of flights enriched

Enriched dataset:
  - Flights with both timezones: 2,125,312 (99.28%)
  - New columns: 6 (timezone, latitude, longitude for both airports)
  - Output path: wingx_timezone.csv


In [11]:
# Match the format you saved in Cell 9
wingx_timezone = pd.read_parquet("wingx_timezone.csv")

In [12]:
wingx_timezone

,FlightDate_utc,Operator,FromAirport,FromState,ToAirport,ToState,Hours,aircraft_tailsign,aircraft_model,aircraft_segment,...,FlightDate_ET,year,month,dow,hour,FromMetro,ToMetro,FromAirport_timezone,FromAirport_latitude,FromAirport_longitude
0,2024-03-10T18:37:00.000Z,Mountain Aviation,KTEX,CO,KBJC,CO,0.683333,N903UP,Citation X,Super Midsize Jet,...,2024-03-10 14:37:00-04:00,2024,3,Sunday,14,Denver,Denver,America/Denver,37.953800,-107.907997
1,2024-03-16T22:58:00.000Z,Mountain Aviation,KJAC,WY,KSDL,AZ,1.400000,N903UP,Citation X,Super Midsize Jet,...,2024-03-16 18:58:00-04:00,2024,3,Saturday,18,Denver,Phoenix Valley,America/Denver,43.607300,-110.737999
2,2024-03-01T22:32:00.000Z,Mountain Aviation,KORL,FL,KCHO,VA,1.516667,N907TX,Citation X,Super Midsize Jet,...,2024-03-01 17:32:00-05:00,2024,3,Friday,17,North Florida,DMV,America/New_York,28.545500,-81.332901
3,2024-03-20T16:09:00.000Z,Mountain Aviation,KFCM,MN,KSDL,AZ,2.533333,N907TX,Citation X,Super Midsize Jet,...,2024-03-20 12:09:00-04:00,2024,3,Wednesday,12,Minneapolis,Phoenix Valley,America/Chicago,44.827202,-93.457100
4,2024-03-22T18:12:00.000Z,Mountain Aviation,KSNA,CA,KSLC,UT,1.300000,N908UP,Citation X,Super Midsize Jet,...,2024-03-22 14:12:00-04:00,2024,3,Friday,14,LA Basin,Denver,America/Los_Angeles,33.675063,-117.869281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2140701,2025-11-11T18:19:00.000Z,Integrated Flight Resources Inc,KPGD,FL,KTYQ,IN,2.266667,N293CK,Phenom 300-E,Light Jet,...,2025-11-11 13:19:00-05:00,2025,11,Tuesday,13,South Florida,Chicago,America/New_York,26.920200,-81.990501
2140702,2025-10-21T20:22:00.000Z,NetJets,KGPC,IN,KMSP,MN,1.333333,N872QS,Citation Longitude,Super Midsize Jet,...,2025-10-21 16:22:00-04:00,2025,10,Tuesday,16,OTHER_METRO,Minneapolis,None,NaN,NaN
2140703,2025-10-03T16:28:00.000Z,Best Jets International,KGPC,IN,KYIP,MI,0.733333,N896BB,Challenger 300,Super Midsize Jet,...,2025-10-03 12:28:00-04:00,2025,10,Friday,12,OTHER_METRO,Detroit,None,NaN,NaN
2140704,2025-10-17T17:23:00.000Z,Northwest Aviation LLC,KFAR,ND,KGPC,IN,1.616667,N415SD,Citation CJ4,Light Jet,...,2025-10-17 13:23:00-04:00,2025,10,Friday,13,Minneapolis,OTHER_METRO,America/Chicago,46.920700,-96.815804


In [13]:
wingx_timezone[wingx_timezone["FromAirport_timezone"].isna()]

,FlightDate_utc,Operator,FromAirport,FromState,ToAirport,ToState,Hours,aircraft_tailsign,aircraft_model,aircraft_segment,...,FlightDate_ET,year,month,dow,hour,FromMetro,ToMetro,FromAirport_timezone,FromAirport_latitude,FromAirport_longitude
217,2024-02-15T18:56:00.000Z,Mile High Aviation Georgia LLC,KCNI,GA,KCNI,GA,0.966667,N77FD,Citation I,Light Jet,...,2024-02-15 13:56:00-05:00,2024,2,Thursday,13,OTHER_METRO,OTHER_METRO,None,NaN,NaN
352,2024-01-28T16:44:00.000Z,CVJ62B LLC,KXNX,TN,KDTS,FL,1.283333,N4047S,Citation M2 Gen2,Light Jet,...,2024-01-28 11:44:00-05:00,2024,1,Sunday,11,OTHER_METRO,Atlanta,None,NaN,NaN
509,2024-03-10T14:01:00.000Z,5M Aviation Holdings LLC,KFHB,FL,KPNS,FL,0.916667,N3JL,Phenom 300,Light Jet,...,2024-03-10 10:01:00-04:00,2024,3,Sunday,10,OTHER_METRO,Atlanta,None,NaN,NaN
674,2024-02-22T12:10:00.000Z,Davis Leasing LLC,KHMP,GA,KVRB,FL,0.933333,N546MD,Citation X,Super Midsize Jet,...,2024-02-22 07:10:00-05:00,2024,2,Thursday,7,OTHER_METRO,South Florida,None,NaN,NaN
726,2024-01-08T18:19:00.000Z,Steel Warehouse Co LLC,KFHB,FL,KSRQ,FL,0.950000,N44SW,no model available,Light Jet,...,2024-01-08 13:19:00-05:00,2024,1,Monday,13,OTHER_METRO,South Florida,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2140294,2025-11-01T00:00:00.000Z,NetJets,KCVC,GA,KTEB,NJ,1.533333,N822QS,Citation Longitude,Super Midsize Jet,...,2025-10-31 20:00:00-04:00,2025,10,Friday,20,OTHER_METRO,New York City,None,NaN,NaN
2140490,2025-11-09T20:51:00.000Z,NetJets,KCNI,GA,KSWF,NY,1.800000,N980QS,no model available,Super Midsize Jet,...,2025-11-09 15:51:00-05:00,2025,11,Sunday,15,OTHER_METRO,New York City,None,NaN,NaN
2140574,2025-10-14T23:03:00.000Z,flyExclusive,KFHB,FL,KPIE,FL,0.700000,N932JS,Citation Sovereign,Super Midsize Jet,...,2025-10-14 19:03:00-04:00,2025,10,Tuesday,19,OTHER_METRO,North Florida,None,NaN,NaN
2140702,2025-10-21T20:22:00.000Z,NetJets,KGPC,IN,KMSP,MN,1.333333,N872QS,Citation Longitude,Super Midsize Jet,...,2025-10-21 16:22:00-04:00,2025,10,Tuesday,16,OTHER_METRO,Minneapolis,None,NaN,NaN


In [14]:

# 1. Ensure FlightDate_utc is datetime
wingx_timezone['FlightDate_utc'] = pd.to_datetime(wingx_timezone['FlightDate_utc'])

# 2. Create the target column initialized with NaT
wingx_timezone['FlightDate_ET_Final'] = pd.NaT

# 3. Vectorized conversion per unique timezone
# This avoids looping over 2 million rows and only loops over ~10 unique timezones
unique_tzs = wingx_timezone['FromAirport_timezone'].dropna().unique()

for tz in unique_tzs:
    mask = wingx_timezone['FromAirport_timezone'] == tz
    # Convert UTC to the specific airport timezone, then convert to US/Eastern
    wingx_timezone.loc[mask, 'FlightDate_ET_Final'] = (
        wingx_timezone.loc[mask, 'FlightDate_utc']
        .dt.tz_convert(tz)
        .dt.tz_convert('US/Eastern')
    )

# 4. Fast fallback: Fill missing values with the existing FlightDate_ET
wingx_timezone['FlightDate_ET_Final'] = wingx_timezone['FlightDate_ET_Final'].fillna(wingx_timezone['FlightDate_ET'])

print("✅ High-speed conversion complete.")

/var/tmp/ipykernel_66412/2226227214.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<DatetimeArray>
['2024-03-10 14:37:00-04:00', '2024-03-16 18:58:00-04:00',
 '2024-03-26 13:16:00-04:00', '2024-03-01 17:37:00-05:00',
 '2024-01-25 17:56:00-05:00', '2024-02-02 20:32:00-05:00',
 '2024-01-22 21:40:00-05:00', '2024-01-02 16:42:00-05:00',
 '2024-01-03 13:32:00-05:00', '2024-01-06 16:16:00-05:00',
 ...
 '2025-10-14 06:00:00-04:00', '2025-10-11 10:04:00-04:00',
 '2025-12-15 17:53:00-05:00', '2025-10-03 17:08:00-04:00',
 '2025-12-07 13:58:00-05:00', '2025-12-29 10:46:00-05:00',
 '2025-10-05 13:31:00-04:00', '2025-11-09 16:52:00-05:00',
 '2025-12-01 17:11:00-05:00', '2025-12-16 15:40:00-05:00']
Length: 166238, dtype: datetime64[ns, US/Eastern]' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  wingx_timezone.loc[mask, 'FlightDate_ET_Final'] = (


✅ High-speed conversion complete.


/var/tmp/ipykernel_66412/2226227214.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  wingx_timezone['FlightDate_ET_Final'] = wingx_timezone['FlightDate_ET_Final'].fillna(wingx_timezone['FlightDate_ET'])


In [19]:
wingx_timezone["month"]

0           3
1           3
2           3
3           3
4           3
           ..
2140701    11
2140702    10
2140703    10
2140704    10
2140705    12
Name: month, Length: 2140706, dtype: int32

In [17]:
# 1. Ensure the column is in datetime format (if not already)
wingx_timezone['FlightDate_ET_Final'] = pd.to_datetime(wingx_timezone['FlightDate_ET_Final'], utc=True).dt.tz_convert('US/Eastern')

# 2. Extract Year, Month, and Hour
wingx_timezone['year_et'] = wingx_timezone['FlightDate_ET_Final'].dt.year
wingx_timezone['month_et'] = wingx_timezone['FlightDate_ET_Final'].dt.month
wingx_timezone['hour_et'] = wingx_timezone['FlightDate_ET_Final'].dt.hour

# 3. Extract Day of Week Name (e.g., Monday, Tuesday)
wingx_timezone['dow_et'] = wingx_timezone['FlightDate_ET_Final'].dt.day_name()

# 4. Extract Day of Week Number (0=Monday, 6=Sunday) if needed for sorting
wingx_timezone['dow_num_et'] = wingx_timezone['FlightDate_ET_Final'].dt.dayofweek

print("✅ Eastern Time features (Year, Month, DOW, Hour) created successfully.")
print(wingx_timezone[['FlightDate_ET_Final', 'year_et', 'month_et', 'dow_et', 'hour_et']].head())

✅ Eastern Time features (Year, Month, DOW, Hour) created successfully.
        FlightDate_ET_Final  year_et  month_et     dow_et  hour_et
0 2024-03-10 14:37:00-04:00     2024         3     Sunday       14
1 2024-03-16 18:58:00-04:00     2024         3   Saturday       18
2 2024-03-01 17:32:00-05:00     2024         3     Friday       17
3 2024-03-20 12:09:00-04:00     2024         3  Wednesday       12
4 2024-03-22 14:12:00-04:00     2024         3     Friday       14


In [21]:
wingx_timezone.to_parquet("wingx_timezone_et.parquet")